In [1]:
### IMPORTS
import pandas as pd
import numpy as np
from backtester.pnl import pnl
from backtester.metrics_old import calc_metrics, return_metrics, infer_ann_factor, _max_drawdown, _equity_curve
from utils.io import load_partitioned_parquet
from utils.paths import make_data_path
from backtester.metrics.performance_report import PerformanceReport

In [2]:
###CONFIG PLEASE
INITAL_CAPITAL = 100000
LEVERAGE = 1.0
DELAY_BARS = 1 # 1-bar execution delay
MAKER_FEE = 0.000200
TAKER_FEE = 0.000550
SYMBOL = 'BTCUSDT'
INTERVAL = 60
START = '01/01/2026'
END = None

In [3]:
### THis needs to be a function too
### LOAD DATA

cols = ['mark_close'] # Whatever I want for the strategy 
path_ohlcv = make_data_path('ohlcv', SYMBOL, INTERVAL)
df_ohlcv = load_partitioned_parquet(path_ohlcv, start=START, end=END, columns=cols)

if df_ohlcv.empty:
    raise ValueError("df_ohlcv loaded empty. Check path or date filters.")

path_funding = make_data_path('funding', SYMBOL)
df_funding = load_partitioned_parquet(path_funding,start=START, end=END)

if df_funding.empty:
    raise ValueError("df_funding loaded empty. Check path or date filters.")

df_merge = df_ohlcv.merge(df_funding, how='left', on='timestamp')
df_merge['fundingRate'] = df_merge['fundingRate'].fillna(0)
df_merge = df_merge.sort_index()

mark_close = df_merge["mark_close"].astype("float64").to_numpy()
funding  = df_merge["fundingRate"].astype("float64").to_numpy()


In [4]:
pos = np.ones(len(mark_close)) # always long lol
print(len(pos))

1486


In [5]:
pos = np.zeros_like(mark_close)
for i in range(len(mark_close)):
    if i % 2 == 0:
        pos[i] = 1

In [6]:
pos = np.random.uniform(-1, 1, size=(len(mark_close)))

In [7]:
pnl_df = pnl(df_merge, pos, capital=INITAL_CAPITAL,leverage=LEVERAGE, delay_bars=DELAY_BARS)
print(pnl_df.head(5))
#print(pnl_df.tail(10))

                           asset_change (%)  signal (% of equity)  \
timestamp                                                           
2026-01-01 00:00:00+00:00          0.000000             -0.934228   
2026-01-01 01:00:00+00:00          0.001777              0.886150   
2026-01-01 02:00:00+00:00         -0.000459              0.967025   
2026-01-01 03:00:00+00:00         -0.000638             -0.952311   
2026-01-01 04:00:00+00:00         -0.003008             -0.290375   

                           held_pos (% of equity)  trade (% of equity)  \
timestamp                                                                
2026-01-01 00:00:00+00:00                0.000000             0.000000   
2026-01-01 01:00:00+00:00               -0.934228            -0.934228   
2026-01-01 02:00:00+00:00                0.886150             1.820379   
2026-01-01 03:00:00+00:00                0.967025             0.080874   
2026-01-01 04:00:00+00:00               -0.952311            -1.919336  

In [8]:
freq, ann = infer_ann_factor(pnl_df.index)
returns = pnl_df['returns_normalised']
metrics = return_metrics(returns, ann, freq)
print(metrics)

{'Frequency': '1H', 'CAGR': -0.9427745653331163, 'Sharpe': -8.99243241305726, 'Sortino': -12.277727864611064, 'Annualised Volatility': 0.3126476426038445, 'Hit Rate (All Bars)': 0.37617765814266485, 'Hit Rate (Trade Bars)': 0.37643097643097645, 'Avg Win/Loss Ratio': 1.1914284076300672, 'Expectancy': np.float64(-0.0003211593648580242), 'Profit Factor': 0.7192316197248461}


In [ ]:
report=PerformanceReport(pnl_df)


8760.0
-2.811462795216753
29.25236835784642
0.22898885088668763
Mean per bar: -0.00032094324146309964
Ann factor: 8760.0
Mean * ann: -2.811462795216753
Final equity: 0.6155225890906426
